# Text Generation for Interpretability

The `irtk.generation` module provides autoregressive generation with full interpretability support:

- **Sampling strategies**: greedy, top-k, nucleus (top-p)
- **Cache-aware generation**: collect activations at every step
- **Model comparison**: generate from two models side by side

Generation is essential for mechinterp because you often want to see how interventions (steering, ablation) affect actual text output, not just single-token logits.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import generation, vis

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. Basic Generation

Greedy generation (temperature=0) for deterministic output.

In [ ]:
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)

# Greedy generation
output = generation.generate(model, tokens, max_new_tokens=20, temperature=0.0)
print(f"Greedy: {tokenizer.decode(np.array(output))}")

# With temperature sampling
output_t = generation.generate(model, tokens, max_new_tokens=20, temperature=0.8, seed=42)
print(f"Temp=0.8: {tokenizer.decode(np.array(output_t))}")

## 2. Sampling Strategies

Top-k and nucleus (top-p) sampling control the diversity of generation.

In [ ]:
prompt = "Once upon a time in a land far away,"
tokens = model.to_tokens(prompt)

# Top-k sampling (only sample from top 50 tokens)
output_topk = generation.generate(
    model, tokens, max_new_tokens=30, temperature=0.9, top_k=50, seed=42
)
print(f"Top-k=50: {tokenizer.decode(np.array(output_topk))}")

# Nucleus sampling (sample from smallest set covering 90% probability)
output_topp = generation.generate(
    model, tokens, max_new_tokens=30, temperature=0.9, top_p=0.9, seed=42
)
print(f"Top-p=0.9: {tokenizer.decode(np.array(output_topp))}")

## 3. Sampling Strategy Internals

Let's visualize what top-k and nucleus sampling actually do to the logit distribution.

In [ ]:
logits = model(tokens)[-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw logits -> probs (sorted)
raw_probs = np.array(jax.nn.softmax(logits))
sorted_probs = np.sort(raw_probs)[::-1][:50]
axes[0].bar(range(len(sorted_probs)), sorted_probs)
axes[0].set_title('Raw probability distribution (top 50)')
axes[0].set_xlabel('Rank')

# After top-k
topk_logits = generation.top_k_sampling(logits, k=10)
topk_probs = np.array(jax.nn.softmax(topk_logits))
topk_sorted = np.sort(topk_probs)[::-1][:50]
axes[1].bar(range(len(topk_sorted)), topk_sorted)
axes[1].set_title('After top-k (k=10)')
axes[1].set_xlabel('Rank')

# After nucleus
nucleus_logits = generation.nucleus_sampling(logits, p=0.9)
nuc_probs = np.array(jax.nn.softmax(nucleus_logits))
nuc_sorted = np.sort(nuc_probs)[::-1][:50]
axes[2].bar(range(len(nuc_sorted)), nuc_sorted)
axes[2].set_title('After nucleus (p=0.9)')
axes[2].set_xlabel('Rank')

plt.tight_layout()
plt.show()

## 4. Generation with Cache

The key feature for interpretability: collect activations at every generation step.
This lets you track how the model's internal representations evolve during generation.

In [ ]:
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)

# Generate with full cache
output, caches = generation.generate_with_cache(
    model, tokens, max_new_tokens=10, temperature=0.0
)

print(f"Generated: {tokenizer.decode(np.array(output))}")
print(f"Collected {len(caches)} caches (one per generation step)")
print(f"Hooks in first cache: {len(caches[0].cache_dict)}")

In [ ]:
# Track attention entropy at layer 11 head 0 across generation steps
from irtk.attention_utils import entropy

entropies = []
for i, cache in enumerate(caches):
    ent = entropy(cache, layer=11, head=0)
    entropies.append(float(ent[-1]))  # entropy at last position

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(entropies)), entropies, 'o-')
ax.set_xlabel('Generation step')
ax.set_ylabel('Attention entropy (L11H0, last pos)')
ax.set_title('Attention Entropy During Generation')
ax.grid(True, alpha=0.3)

# Label generated tokens
prompt_len = len(tokens)
for i in range(len(caches)):
    tok_str = tokenizer.decode([int(output[prompt_len + i])])
    ax.annotate(tok_str, (i, entropies[i]), textcoords="offset points",
                xytext=(0, 10), ha='center', fontsize=7)

plt.tight_layout()
plt.show()

## 5. Selective Caching

For long generation, cache only the hooks you need to save memory.

In [ ]:
# Only cache the final residual stream
output, caches = generation.generate_with_cache(
    model, tokens, max_new_tokens=10, temperature=0.0,
    hook_names=["blocks.11.hook_resid_post"]
)

print(f"Hooks cached per step: {list(caches[0].cache_dict.keys())}")

# Track the norm of the residual stream at the last position
norms = [float(jnp.linalg.norm(c.cache_dict['blocks.11.hook_resid_post'][-1])) for c in caches]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(norms)), norms, 'o-')
ax.set_xlabel('Generation step')
ax.set_ylabel('Residual stream norm (L11, last pos)')
ax.set_title('Residual Stream Norm During Generation')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Stop Token

Stop generation early when a specific token is produced.

In [ ]:
prompt = "Q: What is 2+2? A:"
tokens = model.to_tokens(prompt)

# Stop at newline
newline_token = tokenizer.encode("\n")[0]
output = generation.generate(
    model, tokens, max_new_tokens=50, temperature=0.0, stop_token=newline_token
)
print(f"Output: {tokenizer.decode(np.array(output))}")
print(f"Generated {len(output) - len(tokens)} tokens before stopping")

## Summary

| Function | Purpose |
|----------|--------|
| `generate()` | Basic autoregressive generation |
| `generate_with_cache()` | Generate while caching activations |
| `generate_comparison()` | Compare generation from two models |
| `top_k_sampling()` | Filter logits to top-k |
| `nucleus_sampling()` | Filter logits to top-p cumulative probability |